<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [17]:
# Task type: Ranking/Scoring

# I am treating this as a ranking problem.
# The goal is not just to say “declining/not declining” but to order pages so that a small team sees the highest-priority refresh candidates first.

# This sits between classification and pure scoring: we can train a model that predicts a priority score (or probability of being worth refreshing), then sort by that score.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [18]:
# Target/proxy: is_declining_label (or a continuous priority score derived from it)

# In the starter data the label is defined as:
# is_declining_label=1 if trend_direction=="down" else 0

# This is an observed outcome (the page’s recent traffic trend), not a human-defined rule.

# For ranking we can either:
# predict the binary label and rank by predicted probability, or
# define a continuous proxy that combines declining status with impressions and staleness.

import pandas as pd
!git clone --depth 1 https://github.com/Fatima-05/FlyRank-ML
%cd FlyRank-ML

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(df.shape)
print(df["is_declining_label"].value_counts(normalize=True).round(3))
df[["impressions_90d", "trend_direction", "is_declining_label"]].head()

Cloning into 'FlyRank-ML'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 91 (delta 12), reused 61 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.84 MiB | 13.99 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/FlyRank-ML/FlyRank-ML/FlyRank-ML/FlyRank-ML
(30000, 45)
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


,impressions_90d,trend_direction,is_declining_label
0,3803,down,1
1,15320,down,1
2,12581,down,1
3,11751,stable,0
4,19140,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [19]:
# Primary success metric: Precision@K (especially Precision@50)

# A small team can only review a limited number of pages, so what matters most is how many of the top-K recommended pages are actually declining (or high-priority).

# Precision@50≈0.24 was the hand-written baseline in Notebook 01
# A good model roughly triples that (≈0.68–0.74)

# I will also watch recall of declining pages inside the top 50.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [20]:
# Unit of analysis=one content page (one row=one page)

# Each row in the starter dataset represents a single anonymized page with its search and content signals.

print("Shape:", df.shape)
print("\nOne row = one page. Sample columns:")
display(df[["impressions_90d", "trend_direction", "days_since_last_update",
            "avg_position", "ctr", "word_count"]].head(5))

print("\nTarget sketch:")
display(df[["impressions_90d", "trend_direction", "is_declining_label"]].head(5))

Shape: (30000, 45)

One row = one page. Sample columns:


,impressions_90d,trend_direction,days_since_last_update,avg_position,ctr,word_count
0,3803,down,20,10.6,0.76,3221.0
1,15320,down,25,20.3,0.05,2481.0
2,12581,down,20,36.5,0.09,3515.0
3,11751,stable,22,6.2,0.49,NaN
4,19140,down,14,44.0,0.13,2803.0



Target sketch:


,impressions_90d,trend_direction,is_declining_label
0,3803,down,1
1,15320,down,1
2,12581,down,1
3,11751,stable,0
4,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [21]:
# A simple fixed rule (e.g “if declining and impressions > X and days_since_update > Y”) can only combine a few thresholds

# Real priority is messier:
# High impressions + mild decline can be more valuable than low impressions + strong decline
# Position, CTR gap, and content age interact in non-linear ways
# The best ranking changes when the data distribution shifts

# In Notebook 01 the hand-written rule only reached ~24% Precision@50 while a learned model reached roughly 3× that.
# That gap is exactly why a ranking/scoring model is worth building instead of staying with if-statements.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.